# Sprint 53 · LLM-as-a-Judge experiments on Colab (no Google Drive)

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run cells 1 → 5 in order.
Nothing is kept on Colab after a session ends, so the notebook **downloads a checkpoint zip** to your computer;
next session you upload the newest checkpoint and everything continues where it stopped.

| Cell | What it does | When |
|---|---|---|
| 1 | Check the GPU | every session |
| 2 | Upload your zip(s) and unpack them to `/content/sprint53` | every session |
| 3 | Install + start Ollama, download glm4 and qwen2.5:7b, record the environment | every session (~3–6 min) |
| 4 | Show progress of all 11 phases | every session |
| 5 | Define `save_checkpoint()` (zip + download) | every session |
| 6 | **Step 2** — smoke test: 1 call per prompt per model, parse check + speed estimate | Step 2 only |
| 7 | **Step 4** — run the experiments in 30-min chunks, downloading a checkpoint after each chunk | Step 4 only (not yet) |
| 8 | Manual checkpoint | any time |

In [11]:
# 1. GPU check - should show a Tesla T4 with ~15 GB
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


**Cell 2 — what to upload**
* **First Colab session:** `sprint53_from_lightning.zip` (reviewed Step 1 folder) **and** `sprint53_step2_code.zip`.
* **Every later session:** only the newest `sprint53_checkpoint_<date>.zip` from the Downloads folder.

We can select several files in the upload dialog at once. Zips are unpacked in a safe order
(data/checkpoint first, code last), so the newest code always wins and results are never overwritten by an older code zip.

In [12]:
# 2. Upload and unpack - asks for files one at a time until everything needed is present
import os, zipfile, glob
from google.colab import files
os.chdir('/content')
code_zips = []

def unpack(path):
    with zipfile.ZipFile(path) as z:
        tops = {p.split('/')[0] for p in z.namelist() if p.strip('/')}
        if tops != {'sprint53'}:
            raise SystemExit(f'!! {path} does not contain a single sprint53/ folder (found {tops}). Re-zip the folder itself.')
        z.extractall('/content')
    print(f'   unpacked {path}')

def have_data():
    return os.path.exists('/content/sprint53/data/processed/setB_items.jsonl')

def have_code():
    return os.path.exists('/content/sprint53/run_experiments.py')

while not (have_data() and have_code()):
    need = []
    if not have_data(): need.append('Step 1 data (sprint53_from_lightning.zip, or your newest checkpoint zip)')
    if not have_code(): need.append('Step 2 code (sprint53_step2_code.zip)')
    print('Please upload ONE file now:', ' / '.join(need))
    for name in files.upload():
        if 'code' in name.lower():
            code_zips.append(name)
            unpack(name)
        else:
            unpack(name)
            for c in code_zips:      # re-apply code uploaded earlier, so newer code always wins
                unpack(c)

os.chdir('/content/sprint53')
print('\nReady. Contents:', sorted(os.listdir('.')))
print('Results files:', sorted(os.path.basename(p) for p in glob.glob('results/*')) or 'none yet')


Ready. Contents: ['LIGHTNING_GUIDE.md', 'README_STEP1.md', 'README_STEP2.md', 'colab', 'data', 'judge', 'notebooks', 'package.json', 'requirements.txt', 'results', 'review', 'run_experiments.py', 'run_step1.sh', 'scripts', 'step1_final_data.zip', 'tests']
Results files: ['P01_E1_T00.jsonl', 'P02_E1_T07.jsonl', 'P03_E1_T10.jsonl', 'P04_E2_prompts.jsonl', 'P05_E3_scales.jsonl', 'P06_E4_setB_mqm.jsonl', 'P07_E4_mqm_ref.jsonl', 'environment.jsonl', 'progress.json', 'prompts_snapshot.json', 'smoke.jsonl', 'smoke_summary.json']


In [13]:
!python run_experiments.py --status
!ls -lt /content/*.zip | head -3

phase                exp      calls   done     % malformed
P01_E1_T00           E1         480    480  100%         0  ✔
P02_E1_T07           E1         800    800  100%         0  ✔
P03_E1_T10           E1         800    800  100%         0  ✔
P04_E2_prompts       E2         320    320  100%         0  ✔
P05_E3_scales        E3         160    160  100%         0  ✔
P06_E4_setB_mqm      E4-E7      550    550  100%         0  ✔
P07_E4_mqm_ref       E4         630    105   17%         0  …
P08_E5_setB_flat     E5-E7      550      0    0%         0   
P09_E8_script        E8         148      0    0%         0   
P10_E9_pairwise      E9         240      0    0%         0   
P11_E10_rationale    E10         80      0    0%         0   
TOTAL                          4758   3215   68%
Next phase to run: P07_E4_mqm_ref
-rw-r--r-- 1 root root 651114 Sep 30 13:19 /content/sprint53_checkpoint_20260930_1319.zip
-rw-r--r-- 1 root root 651112 Sep 30 13:18 /content/sprint53_checkpoint_20260930_1318.

In [ ]:
import json
rows = [json.loads(l) for l in open('data/processed/setB_items.jsonl')]
print('rows:', len(rows))
print('MT outputs kept with reviewer fix:', sum(1 for r in rows if r.get('reviewer_suggested_correction')))
print('S52_020 gold:', next(r['candidate'] for r in rows if r['item_id'] == 'B::S52_020::gold'))

rows: 275
MT outputs kept with reviewer fix: 5
S52_020 gold: 说得客气一点，首席财务官的那番话不太坦诚。


In [18]:
# 3. Install + start Ollama, download both judge models, record GPU/Ollama/model versions
%cd /content/sprint53
!bash colab/setup_ollama.sh
!python run_experiments.py --check

/content
>> 1/4 System packages (zstd is needed by the Ollama installer, pciutils/lshw for GPU detection)
>> 2/4 Ollama already installed
>> 3/4 Starting the Ollama server in the background (log: /content/ollama.log)
{"version":"0.35.0"}
>> 4/4 Downloading judge models (same tags as Sprint 50)
success 
success 
NAME           ID              SIZE      MODIFIED               
qwen2.5:7b     845dbda0ea48    4.7 GB    Less than a second ago    
glm4:latest    5b699761eca5    5.5 GB    Less than a second ago    
>> Setup finished.
GPU            : Tesla T4, 15360 MiB
Ollama version : 0.35.0
Model glm4     : glm4:latest  (digest 5b699761eca5)
Model qwen     : qwen2.5:7b  (digest 845dbda0ea48)
OK - environment recorded in results/environment.jsonl


In [19]:
# 4. Progress of every phase (expected before Step 4: 11 phases, TOTAL 4758, 0 done)
!python run_experiments.py --status

phase                exp      calls   done     % malformed
P01_E1_T00           E1         480    480  100%         0  ✔
P02_E1_T07           E1         800    800  100%         0  ✔
P03_E1_T10           E1         800    800  100%         0  ✔
P04_E2_prompts       E2         320    320  100%         0  ✔
P05_E3_scales        E3         160    160  100%         0  ✔
P06_E4_setB_mqm      E4-E7      550    550  100%         0  ✔
P07_E4_mqm_ref       E4         630    630  100%         0  ✔
P08_E5_setB_flat     E5-E7      550    550  100%         0  ✔
P09_E8_script        E8         148     72   49%         0  …
P10_E9_pairwise      E9         240      0    0%         0   
P11_E10_rationale    E10         80      0    0%         0   
TOTAL                          4758   4362   92%
Next phase to run: P09_E8_script


In [16]:
# 5. Checkpoint helper: zips /content/sprint53 (data, code, results - not the models) and downloads it
import datetime, zipfile, os
from google.colab import files
def save_checkpoint():
    stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M')
    out = f'/content/sprint53_checkpoint_{stamp}.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, dirs, fnames in os.walk('/content/sprint53'):
            dirs[:] = [d for d in dirs if d not in ('node_modules', '.venv', '__pycache__')]
            for f in fnames:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, '/content'))
    print(f'Checkpoint: {out} ({os.path.getsize(out)/1e6:.1f} MB) - downloading...')
    files.download(out)
print('save_checkpoint() is ready')

save_checkpoint() is ready


## STEP 2 — smoke test
24 calls (12 prompt templates × 2 models) on fixed items. Checks that every prompt works with both judges,
that the answers can be parsed, and measures real speed on this GPU. Takes ~2–5 minutes.
It then downloads a checkpoint so you keep the smoke results.

In [ ]:
# 6. Smoke test (Step 2)
!python run_experiments.py --smoke
save_checkpoint()

>> Smoke test: 24 calls (12 prompts x 2 models). The first call per model is slower (model loading).
   [smoke] 24/24 calls  |  7.9 s/call  |  phase ETA 0m00s  |  session time left 56m50s

model  prompt               parse         value tries   secs tokens
glm4   flat                 ok               85     1  103.3     12
glm4   mqm                  ok               85     1    3.4    115
glm4   mqm_reworded         ok               85     1    3.5    115
glm4   mqm_zh               ok               90     1    2.0     63
glm4   mqm_severity_defs    ok               85     1    3.5    115
glm4   mqm_analysis_first   ok               98     1    5.2    168
glm4   flat_1to5            ok                3     1    0.4     12
glm4   flat_1to10           ok                7     1    0.4     12
glm4   mqm_ref              ok               85     1    3.5    111
glm4   mqm_neutral_label    ok               75     1    1.6     48
glm4   pairwise             ok              tie     1    0.4   

/tmp/ipykernel_1329/15651886.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## STEP 4 — run the experiments  ⚠️ do NOT run until Step 3 is confirmed
Runs the next unfinished phase(s) in 30-minute chunks and downloads a checkpoint after every chunk, until
`SESSION_BUDGET` is used up or everything is finished. If Colab disconnects, you lose at most the current chunk's
unsaved calls — upload the newest checkpoint next session and run this cell again.
Keep this browser tab open (the checkpoint downloads need it). Allow multiple downloads if Chrome asks.

In [20]:
# 7. Run experiments (Step 4)
import time, json
SESSION_BUDGET_MIN = 195     # 3h15m - stop well before a ~3.5 h free session ends
CHUNK_MIN = 30
WORKERS = 1                  # decided in Step 3
start = time.time()
while True:
    left = SESSION_BUDGET_MIN - (time.time() - start) / 60
    if left < 3:
        print('Session budget used - stop here, keep the last checkpoint, continue next session.'); break
    chunk = int(min(CHUNK_MIN, left))
    get_ipython().system(f'python run_experiments.py --run --budget {chunk}m --workers {WORKERS}')
    rc = get_ipython().user_ns.get('_exit_code', 0)
    save_checkpoint()
    if rc != 0:
        print('!! The runner stopped with an error (see above). Saved calls are in the checkpoint. '
              'Re-run cell 3 to restart Ollama, then run this cell again.'); break
    prog = json.load(open('results/progress.json'))
    if prog['next_phase'] is None:
        print('ALL PHASES COMPLETE'); break

>> Session budget 30m00s, 1 worker(s), models: glm4=glm4:latest, qwen=qwen2.5:7b
== P01_E1_T00: already complete (480 calls)
== P02_E1_T07: already complete (800 calls)
== P03_E1_T10: already complete (800 calls)
== P04_E2_prompts: already complete (320 calls)
== P05_E3_scales: already complete (160 calls)
== P06_E4_setB_mqm: already complete (550 calls)
== P07_E4_mqm_ref: already complete (630 calls)
== P08_E5_setB_flat: already complete (550 calls)
== P09_E8_script (E8): 72/148 done, 76 to go
   [P09_E8_script] 25/76 calls  |  3.2 s/call  |  phase ETA 2m42s  |  session time left 28m40s
   [P09_E8_script] 50/76 calls  |  2.3 s/call  |  phase ETA 0m59s  |  session time left 28m05s
   [P09_E8_script] 75/76 calls  |  2.0 s/call  |  phase ETA 0m01s  |  session time left 27m30s
   [P09_E8_script] 76/76 calls  |  2.0 s/call  |  phase ETA 0m00s  |  session time left 27m30s
   P09_E8_script: +76 calls this session, 2.0 s/call
== P10_E9_pairwise (E9): 0/240 done, 240 to go
   [P10_E9_pairwise]

/tmp/ipykernel_1901/15651886.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ALL PHASES COMPLETE


In [21]:
# 8. Manual checkpoint (run before closing the tab)
save_checkpoint()

Checkpoint: /content/sprint53_checkpoint_20260930_1358.zip (0.7 MB) - downloading...


/tmp/ipykernel_1901/15651886.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
   !unzip -l /content/sprint53_checkpoint_20260930_1358.zip | grep "results/P"

   102783  2026-09-30 12:51   sprint53/results/P05_E3_scales.jsonl
   462404  2026-09-30 12:50   sprint53/results/P04_E2_prompts.jsonl
   696992  2026-09-30 13:36   sprint53/results/P07_E4_mqm_ref.jsonl
   487577  2026-09-30 12:35   sprint53/results/P01_E1_T00.jsonl
   649324  2026-09-30 13:12   sprint53/results/P06_E4_setB_mqm.jsonl
   181183  2026-09-30 13:48   sprint53/results/P09_E8_script.jsonl
   192373  2026-09-30 13:49   sprint53/results/P10_E9_pairwise.jsonl
   814368  2026-09-30 12:35   sprint53/results/P02_E1_T07.jsonl
   360053  2026-09-30 13:40   sprint53/results/P08_E5_setB_flat.jsonl
   806518  2026-09-30 12:35   sprint53/results/P03_E1_T10.jsonl
    88883  2026-09-30 13:52   sprint53/results/P11_E10_rationale.jsonl
